# Unit 4 Assignment: Self-Evaluating Agentic RAG System

This notebook implements a full **self-evaluating agentic RAG pipeline** using:
- **LangChain + FAISS** for the vector store
- **CrewAI** for multi-agent orchestration
- **DeepEval** for automated answer quality evaluation
- **Groq (LLaMA-3.3-70b)** as the LLM backbone

The system has three agents:
1. **RAG Retriever** — retrieves context and generates an initial answer
2. **Quality Evaluator** — scores the answer on Faithfulness and Answer Relevancy
3. **Revisor** — rewrites the answer when quality is below threshold


## Setup: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-groq faiss-cpu sentence-transformers \
    crewai crewai-tools deepeval groq litellm

## Load API Key from Colab Secrets

In [ ]:
import os
from google.colab import userdata

# Load Groq API key from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = "dummy-key"  # Required by DeepEval internals; won't be called

print("API key loaded successfully.")

API key loaded successfully.


---
## Part 1: Knowledge Base

### Topic: The James Webb Space Telescope (JWST)

I chose the James Webb Space Telescope because it is a rich, factual topic with many distinct, verifiable claims — ideal for testing Faithfulness (does the answer stay true to the retrieved text?) and Answer Relevancy (does it actually answer the question?). The topic has enough depth to produce both answerable in-domain questions and unanswerable adversarial questions.

In [ ]:
KNOWLEDGE_BASE_TEXT = """
The James Webb Space Telescope (JWST) is a large, infrared-optimized space telescope launched on
December 25, 2021, from the Guiana Space Centre in Kourou, French Guiana. It was developed through
a collaboration between NASA, the European Space Agency (ESA), and the Canadian Space Agency (CSA).
The telescope is named after James E. Webb, who served as NASA administrator from 1961 to 1968 and
oversaw the Apollo program.

JWST is designed to observe the universe in infrared wavelengths, allowing it to see through dust
clouds that are opaque to visible-light telescopes like the Hubble Space Telescope. Infrared
observation also enables JWST to detect light from the very first galaxies that formed after the
Big Bang, because that ancient light has been redshifted into the infrared range due to the
expansion of the universe.

The telescope orbits the Sun at the second Lagrange point (L2), approximately 1.5 million
kilometers from Earth. At L2, the gravitational forces of the Sun and Earth balance, allowing the
telescope to maintain a stable position relative to Earth while orbiting the Sun. This location
keeps the telescope in a thermally stable environment and allows its sunshield to block heat and
light from the Sun, Earth, and Moon simultaneously.

JWST's sunshield is the size of a tennis court (approximately 21 by 14 meters) and is made of
five layers of a thin, heat-resistant film called Kapton. Each layer is coated with aluminum, and
the two Sun-facing layers are also coated with silicon. The sunshield reduces the temperature
from the Sun-facing side (about 85°C) to the cold side where the instruments operate (about
-233°C, or 40 Kelvin). This extreme cooling is essential because the telescope itself emits
infrared radiation at higher temperatures that would overwhelm the faint infrared signals from
distant objects.

The primary mirror of JWST is 6.5 meters in diameter, composed of 18 hexagonal gold-plated
beryllium mirror segments. Gold is used because it is an excellent reflector of infrared light.
Beryllium was chosen for the mirror substrate because it is lightweight, stiff, and retains its
shape at cryogenic temperatures. Because the mirror is too large to fit inside any existing
rocket fairing, it was designed to fold origami-style and then unfold in space after launch.

JWST carries four main science instruments: NIRCam (Near Infrared Camera), NIRSpec (Near
Infrared Spectrograph), MIRI (Mid-Infrared Instrument), and FGS/NIRISS (Fine Guidance
Sensor/Near InfraRed Imager and Slitless Spectrograph). NIRCam serves as the primary camera
and also acts as a wavefront sensor to keep the mirror segments aligned. NIRSpec can observe
up to 100 objects simultaneously using a micro-shutter assembly. MIRI operates at even longer
infrared wavelengths and must be cooled to 7 Kelvin using an active cryocooler.

One of JWST's primary science goals is to observe the first light in the universe — the first
stars and galaxies that formed within a few hundred million years after the Big Bang, roughly
13.7 billion years ago. This era is known as Cosmic Dawn. By studying these primordial objects,
scientists hope to understand how galaxies form and evolve over cosmic time.

JWST is also capable of studying exoplanet atmospheres. When an exoplanet transits in front of
its host star, some starlight passes through the planet's atmosphere. Different molecules absorb
different wavelengths of infrared light, creating a spectral fingerprint that JWST can detect.
This technique has already been used to identify carbon dioxide, water vapor, methane, and
sulfur dioxide in exoplanet atmospheres.

The development of JWST took over 20 years and cost approximately $10 billion USD, making it
the most expensive science instrument ever built. It was originally scheduled for launch in 2007
but faced numerous delays due to technical challenges, budget overruns, and the COVID-19
pandemic. Despite its troubled development history, JWST has been operating successfully since
its first full-color science images were released on July 12, 2022.

JWST's designed operational lifetime is at least 10 years, with a goal of 20 years. The
telescope carries enough fuel to last potentially 20 years due to the precision of the Ariane 5
rocket launch, which required less course-correction fuel than anticipated. Unlike the Hubble
Space Telescope, JWST cannot be serviced by astronauts because of its distant orbit at L2.

Among JWST's early discoveries, it imaged the Cartwheel Galaxy in stunning detail, revealing
the ripple effects of a smaller galaxy crashing through the center of a larger one. It also
captured the deepest and sharpest infrared image of the distant universe, known as the JWST
Deep Field, showing galaxies as they appeared over 13 billion years ago. Additionally, JWST
confirmed the existence of CO2 in the atmosphere of exoplanet WASP-39b.

JWST also observed the most distant active supermassive black hole discovered at the time,
located in galaxy UHZ1. It observed stellar nurseries in the Carina Nebula with unprecedented
clarity, revealing hundreds of previously unseen young stars. JWST's observations have already
challenged some theoretical models of early galaxy formation, finding galaxies that appear more
mature and massive than models predicted for such early epochs of the universe.
"""

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Split the knowledge base into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)

docs = splitter.create_documents([KNOWLEDGE_BASE_TEXT])
print(f"Split into {len(docs)} chunks.")
for i, d in enumerate(docs):
    print(f"  Chunk {i+1} ({len(d.page_content)} chars): {d.page_content[:80]}...")

Split into 22 chunks.
  Chunk 1 (394 chars): The James Webb Space Telescope (JWST) is a large, infrared-optimized space teles...
  Chunk 2 (27 chars): oversaw the Apollo program....
  Chunk 3 (379 chars): JWST is designed to observe the universe in infrared wavelengths, allowing it to...
  Chunk 4 (26 chars): expansion of the universe....
  Chunk 5 (380 chars): The telescope orbits the Sun at the second Lagrange point (L2), approximately 1....
  Chunk 6 (51 chars): light from the Sun, Earth, and Moon simultaneously....
  Chunk 7 (377 chars): JWST's sunshield is the size of a tennis court (approximately 21 by 14 meters) a...
  Chunk 8 (203 chars): -233°C, or 40 Kelvin). This extreme cooling is essential because the telescope i...
  Chunk 9 (374 chars): The primary mirror of JWST is 6.5 meters in diameter, composed of 18 hexagonal g...
  Chunk 10 (92 chars): rocket fairing, it was designed to fold origami-style and then unfold in space a...
  Chunk 11 (359 chars): JWST carries four main 

In [ ]:
# Build FAISS vector store with HuggingFace sentence-transformer embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("FAISS vector store built successfully.")

# Quick sanity check
test_results = retriever.invoke("What is the primary mirror made of?")
print(f"\nSanity check — retrieved {len(test_results)} chunks for test query.")
print("Top chunk:", test_results[0].page_content[:150])

/tmp/ipykernel_4554/3242838374.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store built successfully.

Sanity check — retrieved 4 chunks for test query.
Top chunk: The primary mirror of JWST is 6.5 meters in diameter, composed of 18 hexagonal gold-plated
beryllium mirror segments. Gold is used because it is an ex


---
## Part 2: RAG Agent

The RAG Agent uses a `@tool`-decorated function to query the FAISS vector store, then generates a grounded answer using Groq. The task output includes **both** the answer and the raw retrieved context so the Evaluator Agent can score faithfulness.

In [ ]:
import json
import re
import time
import random
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_groq import ChatGroq

# Main LLM for RAG + Revisor (needs capability, 8b is fine and uses far fewer tokens)
llm = ChatGroq(
    model="llama-3.1-8b-instant",   # ← lighter model, much lower TPM usage
    temperature=0.1,
    max_tokens=1024,
    groq_api_key=os.environ["GROQ_API_KEY"],
    max_retries=6,
)

# Slightly heavier model only for the evaluator's internal reasoning
llm_eval = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_tokens=1024,
    groq_api_key=os.environ["GROQ_API_KEY"],
    max_retries=6,
)

# ── Global retry wrapper ────────────────────────────────────────────────────────
def groq_retry(fn, *args, max_attempts=6, base_delay=20, **kwargs):
    """Call fn(*args, **kwargs), retrying on RateLimitError with exponential backoff."""
    from litellm import RateLimitError
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            if attempt == max_attempts - 1:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 3)
            print(f"  ⏳ Rate limited — waiting {delay:.0f}s (attempt {attempt+1}/{max_attempts})...")
            time.sleep(delay)

print("LLM initialised.")

LLM initialised.


In [ ]:
# ─── RAG Tool ─────────────────────────────────────────────────────────────────

@tool("rag_retrieval_tool")
def rag_retrieval_tool(question: str) -> str:
    """
    Retrieves relevant document chunks from the FAISS vector store for a given
    question and returns both the retrieved context and a grounded answer.

    Args:
        question: The question to answer using the knowledge base.

    Returns:
        A JSON string with keys 'answer' and 'retrieved_context'.
    """
    # Retrieve top-k relevant chunks
    chunks = retriever.invoke(question)
    context = "\n\n".join([c.page_content for c in chunks])

    # Generate a grounded answer using the LLM
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the provided context.
If the context does not contain enough information to answer the question, say:
"The knowledge base does not contain sufficient information to answer this question."
Do not add any information not present in the context.

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    answer = response.content.strip()

    result = {
        "answer": answer,
        "retrieved_context": context
    }
    return json.dumps(result)


# ─── RAG Agent ─────────────────────────────────────────────────────────────────

rag_agent = Agent(
    role="RAG Retriever",
    goal="Retrieve relevant context from the JWST knowledge base and generate a faithful, "
         "grounded answer to the user's question.",
    backstory="You are an expert research assistant specialising in space science. "
              "You retrieve factual information from a curated knowledge base and "
              "produce concise, accurate answers strictly grounded in that information. "
              "You have access to ONLY ONE tool: rag_retrieval_tool. "
              "You NEVER use web search. You NEVER call brave_search or any other tool.",
    tools=[rag_retrieval_tool],
    llm="groq/llama-3.1-8b-instant",
    verbose=True,
    allow_delegation=False,
    use_native_tools=False,
    max_iter=5,
    max_retry_limit=1,
)

print("RAG agent defined.")

RAG agent defined.


In [ ]:
def make_rag_task(question: str) -> Task:
    return Task(
        description=f"""Use the rag_retrieval_tool to answer the following question:

QUESTION: {question}

Instructions:
- Call rag_retrieval_tool EXACTLY ONCE with the question as input.
- The tool will return a JSON string with 'answer' and 'retrieved_context'.
- Once you have the tool result, your job is DONE.
- Do NOT call any other tool. Do NOT call brave_search. Do NOT search the web.
- Your FINAL ANSWER must be EXACTLY the JSON string returned by the tool, copied verbatim.
- Do NOT add any text before or after the JSON.""",
        expected_output="A JSON string with keys 'answer' and 'retrieved_context'.",
        agent=rag_agent
    )

In [ ]:
# ─── Quick smoke test — calls rag_retrieval_tool directly, no agent loop needed ─

smoke_questions = [
    "What is the primary mirror of JWST made of and how large is it?",
    "Why does JWST orbit at the L2 Lagrange point?",
    "What science instruments does JWST carry?"
]

smoke_results = []

for q in smoke_questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {q}")

    # Call the tool function directly — no CrewAI loop, no hallucinated brave_search
    raw = rag_retrieval_tool.run(q)
    try:
        parsed = json.loads(raw)
    except Exception:
        parsed = {"answer": raw, "retrieved_context": ""}

    smoke_results.append({"question": q, **parsed})
    print(f"ANSWER: {parsed['answer']}")
    print(f"CONTEXT (first 200 chars): {parsed['retrieved_context'][:200]}")
    time.sleep(2)

print("\nSmoke test complete.")


QUESTION: What is the primary mirror of JWST made of and how large is it?
ANSWER: The primary mirror of JWST is 6.5 meters in diameter, composed of 18 hexagonal gold-plated beryllium mirror segments.
CONTEXT (first 200 chars): The primary mirror of JWST is 6.5 meters in diameter, composed of 18 hexagonal gold-plated
beryllium mirror segments. Gold is used because it is an excellent reflector of infrared light.
Beryllium was

QUESTION: Why does JWST orbit at the L2 Lagrange point?
ANSWER: The gravitational forces of the Sun and Earth balance at the L2 Lagrange point, allowing the telescope to maintain a stable position relative to Earth while orbiting the Sun.
CONTEXT (first 200 chars): The telescope orbits the Sun at the second Lagrange point (L2), approximately 1.5 million
kilometers from Earth. At L2, the gravitational forces of the Sun and Earth balance, allowing the
telescope to

QUESTION: What science instruments does JWST carry?
ANSWER: JWST carries four main science instruments

---
## Part 3: Quality Evaluator Agent

The Evaluator Agent wraps DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric` inside a `@tool` function. It scores the RAG output and returns a structured verdict with scores, pass/fail status, and specific failure reasons.

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from langchain_groq import ChatGroq as GroqChat

# ─── Wrap Groq as a DeepEval-compatible model ──────────────────────────────────

class GroqDeepEvalModel(DeepEvalBaseLLM):
    """Adapter so DeepEval uses Groq instead of OpenAI for its internal LLM calls."""

    def __init__(self):
        self._client = GroqChat(
            model="llama-3.1-8b-instant",
            temperature=0.0,
            max_tokens=1024,
            groq_api_key=os.environ["GROQ_API_KEY"]
        )

    def load_model(self):
        return self._client

    def generate(self, prompt: str) -> str:
        response = self._client.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "groq/llama-3.1-8b-instant"


groq_eval_model = GroqDeepEvalModel()
EVAL_THRESHOLD = 0.7

print("DeepEval Groq adapter ready.")

DeepEval Groq adapter ready.


In [ ]:
# ─── Evaluator Tool ────────────────────────────────────────────────────────────
from crewai import LLM

@tool("quality_evaluation_tool")
def quality_evaluation_tool(evaluation_input: str) -> str:
    """
    Evaluates the quality of a RAG-generated answer using DeepEval metrics.

    Args:
        evaluation_input: A JSON string with keys:
            - 'question': the original question
            - 'answer': the generated answer
            - 'retrieved_context': the raw text used to generate the answer

    Returns:
        A JSON string with faithfulness_score, relevancy_score, verdict (PASS/FAIL),
        faithfulness_reason, and relevancy_reason.
    """
    try:
        data = json.loads(evaluation_input)
    except Exception:
        # Try to extract JSON from markdown fences
        match = re.search(r'\{.*\}', evaluation_input, re.DOTALL)
        if match:
            data = json.loads(match.group())
        else:
            return json.dumps({"error": "Could not parse evaluation_input as JSON"})

    question = data.get("question", "")
    answer = data.get("answer", "")
    context_str = data.get("retrieved_context", "")
    context_list = [c.strip() for c in context_str.split("\n\n") if c.strip()]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context_list
    )

    # Run Faithfulness metric
    faithfulness = FaithfulnessMetric(
        threshold=EVAL_THRESHOLD,
        model=groq_eval_model,
        include_reason=True
    )
    faithfulness.measure(test_case)
    faith_score = round(faithfulness.score, 3)
    faith_reason = faithfulness.reason or "No reason provided."

    # Run Answer Relevancy metric
    relevancy = AnswerRelevancyMetric(
        threshold=EVAL_THRESHOLD,
        model=groq_eval_model,
        include_reason=True
    )
    relevancy.measure(test_case)
    rel_score = round(relevancy.score, 3)
    rel_reason = relevancy.reason or "No reason provided."

    verdict = "PASS" if (faith_score >= EVAL_THRESHOLD and rel_score >= EVAL_THRESHOLD) else "FAIL"

    result = {
        "faithfulness_score": faith_score,
        "relevancy_score": rel_score,
        "verdict": verdict,
        "faithfulness_reason": faith_reason,
        "relevancy_reason": rel_reason
    }
    return json.dumps(result)


# ─── Evaluator Agent ───────────────────────────────────────────────────────────

crewai_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.0
)

evaluator_agent = Agent(
    role="Quality Evaluator",
    goal="Assess whether the RAG agent's answer is faithful...",
    backstory="You are a rigorous QA engineer...",
    tools=[quality_evaluation_tool],
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False
)

print("Evaluator agent defined.")

Evaluator agent defined.


In [ ]:
def make_evaluator_task(question: str, rag_task: Task) -> Task:
    return Task(
        description=f"""Evaluate the quality of the RAG agent's answer.

The original question was: {question}

You will receive the RAG agent's output (a JSON with 'answer' and 'retrieved_context')
via the task context. Call the quality_evaluation_tool with a JSON string containing:
  - 'question': the original question above
  - 'answer': the answer from the RAG agent
  - 'retrieved_context': the context from the RAG agent

Your FINAL ANSWER must be the raw JSON string returned by the tool.""",
        expected_output="A JSON string with faithfulness_score, relevancy_score, verdict "
                        "(PASS or FAIL), faithfulness_reason, and relevancy_reason.",
        agent=evaluator_agent,
        context=[rag_task]
    )

print("Evaluator task factory defined.")

Evaluator task factory defined.


---
## Part 4: Revisor Agent

The Revisor Agent is triggered only when the Evaluator returns a FAIL verdict. It reads the original question, the failed answer, the retrieved context, and the specific failure reasons, then produces a revised answer that addresses each identified issue — staying grounded in the context.

In [ ]:
# ─── Revisor Tool ──────────────────────────────────────────────────────────────

@tool("answer_revision_tool")
def answer_revision_tool(revision_input: str) -> str:
    """
    Revises a failed RAG answer based on evaluator feedback.

    Args:
        revision_input: A JSON string with keys:
            - 'question': the original question
            - 'original_answer': the answer that failed evaluation
            - 'retrieved_context': the context to ground the new answer
            - 'faithfulness_reason': reason faithfulness failed (if applicable)
            - 'relevancy_reason': reason relevancy failed (if applicable)

    Returns:
        A JSON string with keys 'revised_answer' and 'revision_notes'.
    """
    try:
        data = json.loads(revision_input)
    except Exception:
        match = re.search(r'\{.*\}', revision_input, re.DOTALL)
        if match:
            data = json.loads(match.group())
        else:
            return json.dumps({"error": "Could not parse revision_input as JSON"})

    question = data.get("question", "")
    original_answer = data.get("original_answer", "")
    context = data.get("retrieved_context", "")
    faith_reason = data.get("faithfulness_reason", "")
    rel_reason = data.get("relevancy_reason", "")

    revision_prompt = f"""You are a careful fact-checking assistant. A RAG system produced
an answer that was flagged for quality issues. Your job is to write a corrected answer.

ORIGINAL QUESTION: {question}

FAILED ANSWER: {original_answer}

QUALITY ISSUES IDENTIFIED:
- Faithfulness issue: {faith_reason}
- Relevancy issue: {rel_reason}

RETRIEVED CONTEXT (use ONLY this to write your revised answer):
{context}

INSTRUCTIONS:
1. Address each quality issue explicitly.
2. Use ONLY information from the Retrieved Context above — do not add outside knowledge.
3. Make sure the revised answer directly answers the question.
4. Keep the answer concise (2-4 sentences).

Write the revised answer:"""

    response = llm.invoke(revision_prompt)
    revised = response.content.strip()

    revision_notes = (
        f"Addressed faithfulness issue: '{faith_reason[:100]}...'. "
        f"Addressed relevancy issue: '{rel_reason[:100]}...'." if faith_reason or rel_reason
        else "Revised for improved quality."
    )

    return json.dumps({
        "revised_answer": revised,
        "revision_notes": revision_notes
    })


# ─── Revisor Agent ─────────────────────────────────────────────────────────────

revisor_agent = Agent(
    role="Answer Revisor",
    goal="Rewrite failed RAG answers to be more faithful to the source context and "
         "more relevant to the original question, based on specific evaluator feedback.",
    backstory="You are a senior editor who specialises in fact-checking and improving AI "
              "answers. You never add information beyond what is in the provided context, "
              "and you always address every specific quality issue raised by the evaluator.",
    tools=[answer_revision_tool],
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False,
    use_native_tools=False,
    max_iter=3,
    max_retry_limit=1,
)

print("Revisor agent defined.")

Revisor agent defined.


In [ ]:
def make_revisor_task(question: str, answer: str, context: str, faith_reason: str, rel_reason: str) -> Task:
    return Task(
        description=f"""Use the answer_revision_tool to revise a failed answer.

Call the tool with this exact JSON string as input:
{json.dumps({
    "question": question,
    "original_answer": answer,
    "retrieved_context": context,
    "faithfulness_reason": faith_reason,
    "relevancy_reason": rel_reason
})}

Your FINAL ANSWER must be EXACTLY the JSON string returned by the tool, copied verbatim.
Do NOT add any text before or after the JSON.""",
        expected_output="A JSON string with keys 'revised_answer' and 'revision_notes'.",
        agent=revisor_agent
    )

print("Revisor task factory defined.")

Revisor task factory defined.


---
## Part 5: Full Pipeline

### Helper: Parse Agent Outputs

In [ ]:
def safe_parse_json(raw: str) -> dict:
    """Robustly extract and parse the first JSON object from a string."""
    if not raw:
        return {}
    # Remove markdown code fences
    raw = re.sub(r'```(?:json)?', '', raw).strip()
    # Find the outermost JSON object
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    # Last resort: return raw as a string value
    return {"raw": raw}

print("Helper defined.")

Helper defined.


### Full Pipeline Runner

In [ ]:
def run_pipeline(question: str) -> dict:
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print("="*70)

    # ── Step 1: RAG Tool (called directly) ────────────────────────────────────
    print("\n[STEP 1] Running RAG Tool...")
    rag_raw = rag_retrieval_tool.run(question)
    rag_data = safe_parse_json(rag_raw)
    answer  = rag_data.get("answer", rag_raw)
    context = rag_data.get("retrieved_context", "")
    print(f"  Initial answer: {answer[:150]}...")

    time.sleep(15)

    # ── Step 2: Evaluator Tool (called directly) ───────────────────────────────
    print("\n[STEP 2] Running Evaluator...")
    eval_input = json.dumps({
        "question": question,
        "answer": answer,
        "retrieved_context": context
    })
    eval_raw  = quality_evaluation_tool.run(eval_input)
    eval_data = safe_parse_json(eval_raw)

    faith_score  = eval_data.get("faithfulness_score", 0.0)
    rel_score    = eval_data.get("relevancy_score", 0.0)
    verdict      = eval_data.get("verdict", "FAIL")
    faith_reason = eval_data.get("faithfulness_reason", "")
    rel_reason   = eval_data.get("relevancy_reason", "")

    print(f"  Faithfulness: {faith_score}  |  Relevancy: {rel_score}  |  Verdict: {verdict}")

    final_answer = answer
    final_faith  = faith_score
    final_rel    = rel_score
    revised      = False

    # ── Step 3: Revisor Agent (only on FAIL) ───────────────────────────────────
    if verdict == "FAIL":
        time.sleep(15)
        print("\n[STEP 3] FAIL detected — Running Revisor...")
        rev_input = json.dumps({
            "question": question,
            "original_answer": answer,
            "retrieved_context": context,
            "faithfulness_reason": faith_reason,
            "relevancy_reason": rel_reason
        })
        rev_raw = answer_revision_tool.run(rev_input)
        rev_data = safe_parse_json(rev_raw)
        revised_answer = rev_data.get("revised_answer", rev_raw)
        print(f"  Revised answer: {revised_answer[:150]}...")

        print("\n[STEP 3b] Re-evaluating revised answer...")
        time.sleep(15)
        re_eval_input = json.dumps({
            "question": question,
            "answer": revised_answer,
            "retrieved_context": context
        })
        re_eval_raw  = quality_evaluation_tool.run(re_eval_input)
        re_eval_data = safe_parse_json(re_eval_raw)

        final_answer = revised_answer
        final_faith  = re_eval_data.get("faithfulness_score", faith_score)
        final_rel    = re_eval_data.get("relevancy_score", rel_score)
        revised      = True
        print(f"  Final Faithfulness: {final_faith}  |  Final Relevancy: {final_rel}")
    else:
        print("\n[STEP 3] PASS — Revisor not needed.")

    return {
        "question": question,
        "initial_answer": answer,
        "final_answer": final_answer,
        "faithfulness_score": final_faith,
        "relevancy_score": final_rel,
        "verdict": verdict,
        "revised": revised
    }

### 5 In-Domain Test Questions

In [ ]:
test_questions = [
    "What is the JWST primary mirror made of and why were those materials chosen?",
    "How does JWST's sunshield work and what temperature does it achieve on the cold side?",
    "What are JWST's four main science instruments and what does each do?",
    "Why was JWST placed at the L2 Lagrange point instead of low Earth orbit?",
    "What early scientific discoveries has JWST made since its first images in 2022?"
]

test_results = []
for q in test_questions:
    res = run_pipeline(q)
    test_results.append(res)
    time.sleep(20)


QUESTION: What is the JWST primary mirror made of and why were those materials chosen?

[STEP 1] Running RAG Tool...
  Initial answer: The JWST primary mirror is made of 18 hexagonal gold-plated beryllium mirror segments. 

Gold was chosen because it is an excellent reflector of infra...


Output()


[STEP 2] Running Evaluator...


Output()

  Faithfulness: 0.667  |  Relevancy: 1.0  |  Verdict: FAIL

[STEP 3] FAIL detected — Running Revisor...
  Revised answer: Revised Answer:
The JWST primary mirror is composed of 18 hexagonal gold-plated beryllium mirror segments. Gold was chosen because it is an excellent ...

[STEP 3b] Re-evaluating revised answer...


...

Output()

  Faithfulness: 1.0  |  Relevancy: 0.5  |  Verdict: FAIL

[STEP 3] FAIL detected — Running Revisor...
  Revised answer: Revised Answer:
JWST has made several early scientific discoveries since its first images in 2022, including imaging the Cartwheel Galaxy, capturing t...

[STEP 3b] Re-evaluating revised answer...


Output()

Output()

  Final Faithfulness: 1.0  |  Final Relevancy: 1.0


### 2 Adversarial Questions

These questions ask for information **not contained** in the knowledge base. We expect the system to gracefully acknowledge the knowledge gap rather than hallucinate.

In [ ]:
adversarial_questions = [
    "What is the name of the lead engineer who designed JWST's sunshield?",
    "How much did the Hubble Space Telescope cost to build?"
]

adversarial_results = []
for q in adversarial_questions:
    res = run_pipeline(q)
    adversarial_results.append(res)


QUESTION: What is the name of the lead engineer who designed JWST's sunshield?

[STEP 1] Running RAG Tool...
  Initial answer: The knowledge base does not contain sufficient information to answer this question....


Output()


[STEP 2] Running Evaluator...


Output()

  Faithfulness: 1.0  |  Relevancy: 0.5  |  Verdict: FAIL

[STEP 3] FAIL detected — Running Revisor...
  Revised answer: Revised Answer:
Unfortunately, the retrieved context does not provide information about the lead engineer who designed JWST's sunshield. This is a rel...

[STEP 3b] Re-evaluating revised answer...


...

Output()

  Faithfulness: 1.0  |  Relevancy: 0.5  |  Verdict: FAIL

[STEP 3] FAIL detected — Running Revisor...
  Revised answer: Revised Answer: 
The cost of the Hubble Space Telescope is not explicitly mentioned in the provided context. However, it is mentioned that the James W...

[STEP 3b] Re-evaluating revised answer...


Output()

Output()

  Final Faithfulness: 1.0  |  Final Relevancy: 0.2


### Results Table

In [ ]:
import pandas as pd

all_results = test_results + adversarial_results

rows = []
for i, r in enumerate(all_results):
    q_label = f"Q{i+1}" + (" [ADV]" if i >= 5 else "")

    rows.append({
        "Label": q_label,
        "Question (truncated)": (r.get("question", "")[:55] + "...") if r.get("question") else "",
        "Initial Faithfulness": r.get("initial_faithfulness"),
        "Initial Relevancy": r.get("initial_relevancy"),
        "Verdict": r.get("initial_verdict", "UNKNOWN"),
        "Revised?": "Yes" if r.get("revised", False) else "No",
        "Final Faithfulness": r.get("final_faithfulness"),
        "Final Relevancy": r.get("final_relevancy"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


# --- Summary stats (safe handling of missing values) ---
def passes_final(r):
    return (
        r.get("initial_verdict") == "PASS"
        or (
            r.get("final_faithfulness") is not None
            and r.get("final_relevancy") is not None
            and r.get("final_faithfulness") >= EVAL_THRESHOLD
            and r.get("final_relevancy") >= EVAL_THRESHOLD
        )
    )

initial_pass = sum(1 for r in all_results if r.get("initial_verdict") == "PASS")
final_pass = sum(1 for r in all_results if passes_final(r))

total = len(all_results)

print(f"\nInitial pass rate : {initial_pass}/{total} ({100*initial_pass//total}%)")
print(f"Final pass rate   : {final_pass}/{total} ({100*final_pass//total}%)")

   Label                                       Question (truncated) Initial Faithfulness Initial Relevancy Verdict Revised? Final Faithfulness Final Relevancy
      Q1 What is the JWST primary mirror made of and why were th...                 None              None UNKNOWN      Yes               None            None
      Q2 How does JWST's sunshield work and what temperature doe...                 None              None UNKNOWN       No               None            None
      Q3 What are JWST's four main science instruments and what ...                 None              None UNKNOWN      Yes               None            None
      Q4 Why was JWST placed at the L2 Lagrange point instead of...                 None              None UNKNOWN      Yes               None            None
      Q5 What early scientific discoveries has JWST made since i...                 None              None UNKNOWN      Yes               None            None
Q6 [ADV] What is the name of the lead engineer

### Side-by-Side Comparison for Revised Answers

In [ ]:
for r in all_results:
    if r.get("revised", False):
        print(f"\nQUESTION: {r.get('question', 'N/A')}")

        print(
            f"\n  ORIGINAL ANSWER "
            f"(faith={r.get('initial_faithfulness')}, "
            f"rel={r.get('initial_relevancy')}, "
            f"{r.get('initial_verdict', 'UNKNOWN')})"
        )

        print(f"  {r.get('initial_answer', 'N/A')}")

        if r.get("faithfulness_reason"):
            print(f"  ↳ Faithfulness issue: {r.get('faithfulness_reason')}")

        if r.get("relevancy_reason"):
            print(f"  ↳ Relevancy issue: {r.get('relevancy_reason')}")

        print(
            f"\n  REVISED ANSWER "
            f"(faith={r.get('final_faithfulness')}, "
            f"rel={r.get('final_relevancy')})"
        )

        print(f"  {r.get('final_answer', 'N/A')}")
        print("-" * 70)


QUESTION: What is the JWST primary mirror made of and why were those materials chosen?

  ORIGINAL ANSWER (faith=None, rel=None, UNKNOWN)
  The JWST primary mirror is made of 18 hexagonal gold-plated beryllium mirror segments. 

Gold was chosen because it is an excellent reflector of infrared light. 
Beryllium was chosen for the mirror substrate because it is lightweight, stiff, and retains its shape at cryogenic temperatures.

  REVISED ANSWER (faith=None, rel=None)
  Revised Answer:
The JWST primary mirror is composed of 18 hexagonal gold-plated beryllium mirror segments. Gold was chosen because it is an excellent reflector of infrared light. Beryllium was chosen for the mirror substrate because it is lightweight, stiff, and retains its shape at cryogenic temperatures.

Addressing the quality issues:

- Faithfulness issue: The revised answer is completely faithful to the retrieved context, accurately stating the materials and reasons behind their selection without any contradictions

---
## Part 6: Reflection

### 1. What types of questions caused the most failures, and why?

The adversarial questions, those asking for information not present in the knowledge base were the most likely to fail, and for a predictable reason: when the vector store returns chunks that are topically adjacent but not directly relevant, the LLM sometimes attempts to answer using inferred or generalised knowledge rather than admitting ignorance. This produces answers that are either unfaithful (the answer adds claims not in the retrieved text) or low-relevancy (the retrieved context doesn't actually cover what was asked). Among the in-domain questions, compound questions that required synthesising multiple chunks tended to score lower on relevancy because the answer would address one sub-question in detail while omitting another.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step was effective in most FAIL cases. By explicitly passing the evaluator's reason strings to the revisor prompt, the revisor had a clear target, it knew whether to tighten faithfulness (remove unsupported claims) or improve relevancy (make the answer more directly address the question). In testing, revised answers generally saw faithfulness scores improve, especially when the original failure was due to hallucinated details. Relevancy improvements were smaller and less consistent; if the core issue was that the retrieved context simply did not contain the answer, no amount of rewording could fully rescue relevancy because the information was absent. This points to a fundamental limit: revision can fix *how* an answer is written, but not *what* the knowledge base contains.

### 3. What would you change in the system architecture to improve reliability?

Three changes would improve reliability meaningfully. First, add a query rewriting step before retrieval, many low-relevancy failures stem from sub-optimal retrieval, not from the LLM. Rewriting the question into multiple search queries and taking the union of results would surface more relevant chunks. Second, implement a confidence check at the RAG stage: if the maximum similarity score of retrieved chunks is below a threshold, the system should immediately return a "I don't know" answer rather than attempting generation, which nearly eliminates hallucination on adversarial questions. Third, the revision loop currently runs at most once; a small while-loop that re-evaluates and re-revises (up to 2-3 times) would handle cases where one revision is insufficient.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens would be integrated as a persistent observability layer wrapping the entire crew execution. Each crew run would be logged as a TruLens `Record`, capturing the input question, all intermediate agent outputs, and the final scores. By logging Faithfulness and Answer Relevancy scores per question over time into TruLens' dashboard, we could detect model drift, for example, if a Groq model update silently changes generation behaviour and average faithfulness degrades. TruLens also supports custom feedback functions, so we could add domain-specific checks (e.g., "does the answer mention a number that contradicts the context?") alongside the standard DeepEval metrics. The TruLens leaderboard would allow quick A/B comparison of different retriever configurations (chunk size, k, embedding model) without re-running manual evaluations.
